# 第二课：线性回归（可视化拟合全过程）

**学习目标**：从零训练线性模型 `y=wx+b`，用图表展示数据、损失曲线、参数收敛、拟合过程。

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import os; os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

## 1. 准备数据（真实关系 y=2x+1 + 噪声）

In [ ]:
np.random.seed(42)
X_np = np.random.uniform(-10, 10, (100,1)).astype(np.float32)
noise = np.random.normal(0, 1.5, (100,1)).astype(np.float32)
y_np = 2.0 * X_np + 1.0 + noise
X = tf.constant(X_np); y_true = tf.constant(y_np)

plt.figure(figsize=(8,5))
plt.scatter(X_np, y_np, alpha=0.6, label='带噪声数据')
xl = np.linspace(-10,10,100)
plt.plot(xl, 2*xl+1, 'r-', lw=2, label='真实 y=2x+1')
plt.xlabel('x'); plt.ylabel('y'); plt.title('训练数据(100样本)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 2. 手动训练循环

In [ ]:
w = tf.Variable(tf.random.normal([1], mean=0, stddev=1.0))
b = tf.Variable(tf.zeros([1]))
lr = 0.01
hist = {'w':[], 'b':[], 'loss':[]}
snaps = []
for epoch in range(200):
    with tf.GradientTape() as tape:
        y_pred = w * X + b
        loss = tf.reduce_mean((y_pred - y_true)**2)
    grads = tape.gradient(loss, [w, b])
    w.assign_sub(lr * grads[0])
    b.assign_sub(lr * grads[1])
    hist['w'].append(w.numpy()[0]); hist['b'].append(b.numpy()[0]); hist['loss'].append(loss.numpy())
    if epoch % 20 == 0:
        snaps.append((epoch, w.numpy()[0], b.numpy()[0]))

### 2.1 损失曲线

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(hist['loss'], 'b-', lw=1.5)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('训练损失曲线 (最终loss=%.4f)'%hist['loss'][-1])
plt.yscale('log'); plt.grid(alpha=0.3); plt.show()

### 2.2 参数收敛轨迹

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].plot(hist['w'], 'b-', lw=1.5, label='学到的w')
axes[0].axhline(2.0, color='r', ls='--', label='真实w=2.0')
axes[0].set_title('w收敛 (最终=%.3f)'%hist['w'][-1])
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_xlabel('Epoch')
axes[1].plot(hist['b'], 'g-', lw=1.5, label='学到的b')
axes[1].axhline(1.0, color='r', ls='--', label='真实b=1.0')
axes[1].set_title('b收敛 (最终=%.3f)'%hist['b'][-1])
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

### 2.3 拟合过程快照（红线逐步贴近绿线）

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18,7))
for idx, (ep, ws, bs) in enumerate(snaps):
    ax = axes.flat[idx]
    ax.scatter(X_np, y_np, alpha=0.3, s=20)
    ax.plot(xl, ws*xl+bs, 'r-', lw=2)
    ax.plot(xl, 2*xl+1, 'g--', lw=1.5, alpha=0.7)
    ax.set_title('Epoch %d\nw=%.2f,b=%.2f'%(ep,ws,bs))
    ax.set_xlim(-10,10); ax.set_ylim(-20,25); ax.grid(alpha=0.3)
axes.flat[0].legend(['数据','模型','真实'], loc='upper left', fontsize=8)
plt.suptitle('拟合过程：红线(模型)逐步贴近绿线(真实)', fontsize=14)
plt.tight_layout(); plt.show()

### 2.4 损失等高线 + 梯度下降轨迹

In [ ]:
wg = np.linspace(-1,5,100); bg = np.linspace(-3,5,100)
W, B = np.meshgrid(wg, bg)
Loss = np.mean((W[...,None]*X_np + B[...,None] - y_np)**2, axis=2)

plt.figure(figsize=(8,6))
cs = plt.contour(W, B, Loss, levels=30, cmap='viridis')
plt.colorbar(cs, label='Loss')
plt.plot(hist['w'], hist['b'], 'ro-', ms=3, lw=1, label='梯度下降轨迹')
plt.scatter([2.0],[1.0], color='yellow', s=200, marker='*', zorder=5, edgecolors='k', label='最优点(2,1)')
plt.scatter([hist['w'][0]],[hist['b'][0]], color='white', s=100, zorder=5, edgecolors='k', label='起点')
plt.xlabel('w'); plt.ylabel('b'); plt.title('损失等高线上的梯度下降轨迹')
plt.legend(); plt.show()

## 3. Keras 版本（5行代码）

In [ ]:
model = tf.keras.Sequential([tf.keras.layers.Dense(1, input_shape=(1,))])
model.compile(optimizer=tf.keras.optimizers.SGD(0.01), loss='mse')
model.fit(X_np, y_np, epochs=200, verbose=0, batch_size=32)
wk, bk = model.layers[0].get_weights()

plt.figure(figsize=(8,5))
plt.scatter(X_np, y_np, alpha=0.3, label='数据')
plt.plot(xl, hist['w'][-1]*xl+hist['b'][-1], 'r-', lw=2,
         label='手动版: w=%.3f,b=%.3f'%(hist['w'][-1],hist['b'][-1]))
plt.plot(xl, wk[0][0]*xl+bk[0], 'g--', lw=2,
         label='Keras版: w=%.3f,b=%.3f'%(wk[0][0],bk[0]))
plt.title('手动实现 vs Keras高层API')
plt.legend(); plt.grid(alpha=0.3); plt.xlabel('x'); plt.ylabel('y')
plt.show()

## 第二课小结

- 训练四步：前向 → 损失 → 梯度 → 更新
- 三大可视化工具：损失曲线、参数轨迹、等高线图
- Keras `model.fit()` 封装了全部循环

**下一课**：CNN 图像分类，可视化特征图。